In [60]:
import pandas as pd
import numpy as np
import os

# --- 1. CONFIGURAZIONE PERCORSI E PARAMETRI ---
base_path = r"C:\SsdRaid\Files\SamuFiles\3anno2sem\Tirocinio\AI_Technical_Debt\analysis"
reports_dir = os.path.join(base_path, "reports", "metrics")
file_path = os.path.join(base_path, "dataset", "all_pr_type.csv")

# NUOVE SOGLIE
ROBUST_THRESHOLD = 10  # Soglia per il report principale (scienza)
MIN_ABSOLUTE = 1       # Soglia minima per non perdere nulla nell'appendice

if not os.path.exists(reports_dir): os.makedirs(reports_dir)

# --- 2. CARICAMENTO E PREPARAZIONE DATI ---
print("Caricamento dataset principale...")
df_main = pd.read_csv(file_path)
df_main['created_at'] = pd.to_datetime(df_main['created_at'])
df_main['merged_at'] = pd.to_datetime(df_main['merged_at'])

# Calcolo tempo e filtro PR "rumore" (durata nulla o negativa)
df_main['time_hrs'] = (df_main['merged_at'] - df_main['created_at']).dt.total_seconds() / 3600
df_main = df_main[(df_main['merged_at'].isna()) | (df_main['time_hrs'] > 0.001)].copy()

# Clipping al 99° percentile per evitare che PR dimenticate aperte distorcano le medie globali
upper_time_limit = df_main['time_hrs'].quantile(0.99)
df_main['time_hrs_clipped'] = df_main['time_hrs'].clip(upper=upper_time_limit)
df_main['is_merged'] = df_main['merged_at'].notna()

df_ai = df_main[df_main['agent'] != 'Human'].copy()
ai_ids = df_ai['id'].unique().tolist()

print("Integrazione metadati granulari...")
df_comm = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_comments.parquet", columns=['pr_id'], filters=[('pr_id', 'in', ai_ids)])
comm_count = df_comm.groupby('pr_id').size().reset_index(name='n_comments')

df_det = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_commit_details.parquet",
                         columns=['pr_id', 'additions', 'deletions', 'commit_stats_total', 'filename'],
                         filters=[('pr_id', 'in', ai_ids)])

# Protezione ASI: calcolo directory e gestione divisioni per zero
df_det['directory'] = df_det['filename'].apply(lambda x: os.path.dirname(str(x)) if pd.notnull(x) else "")
# 1. Raggruppiamo i dati base
pr_stats = df_det.groupby('pr_id').agg({
    'additions': 'sum',
    'deletions': 'sum',
    'commit_stats_total': 'sum',
    'filename': 'nunique' # Conta i file unici (escludendo NaN)
}).reset_index()

# 2. Calcoliamo le directory solo per i file che esistono davvero
dir_stats = df_det[df_det['filename'].notna()].groupby('pr_id')['directory'].nunique().reset_index(name='dir_count')

# 3. Merge e calcolo protetto
pr_stats = pr_stats.merge(dir_stats, on='pr_id', how='left').fillna(0)

# ASI: Se filename è 0, ASI è 0. Altrimenti facciamo il rapporto (max 1.0)
pr_stats['ASI_pr'] = np.where(
    pr_stats['filename'] > 0,
    (pr_stats['dir_count'] / pr_stats['filename']).clip(upper=1.0),
    0
)

# PCD: Densità di modifiche per file
pr_stats['PCD_pr'] = np.where(
    pr_stats['filename'] > 0,
    pr_stats['commit_stats_total'] / pr_stats['filename'],
    0
)

# Merge con how='left' per non perdere PR che non hanno commenti o dettagli commit
df_merged = df_ai.merge(comm_count, left_on='id', right_on='pr_id', how='left') \
                 .merge(pr_stats, left_on='id', right_on='pr_id', how='left').fillna(0)

# --- 3. LOGICA SCR PRECISA (PER REPOSITORY) ---
print("Calcolo SCR granulare (Media delle performance sui repository)...")
df_base_scr = df_ai[['id', 'type', 'agent', 'repo_id', 'merged_at']].dropna(subset=['merged_at'])
df_fix_scr = df_ai[df_ai['type'] == 'fix'][['agent', 'repo_id', 'created_at']]

df_scr_check = df_base_scr.merge(df_fix_scr, on=['agent', 'repo_id'], suffixes=('_task', '_fix'))
df_scr_check['delta'] = (df_scr_check['created_at'] - df_scr_check['merged_at']).dt.total_seconds() / 3600
regressions = df_scr_check[(df_scr_check['delta'] > 0) & (df_scr_check['delta'] <= 48)]

def get_precise_scr(group_col):
    if regressions.empty:
        return pd.DataFrame({group_col: df_base_scr[group_col].unique(), 'SCR': 0.0})

    # Conteggio per singola repo per normalizzare il bias del progetto
    reg_per_repo = regressions.groupby([group_col, 'repo_id']).size().reset_index(name='n_fixes')
    base_per_repo = df_base_scr.groupby([group_col, 'repo_id']).size().reset_index(name='n_tasks')

    scr_repo_stats = base_per_repo.merge(reg_per_repo, on=[group_col, 'repo_id'], how='left').fillna(0)
    scr_repo_stats['scr_local'] = scr_repo_stats['n_fixes'] / scr_repo_stats['n_tasks']

    # Ritorna la media degli SCR locali per ogni agente/tipo
    return scr_repo_stats.groupby(group_col)['scr_local'].mean().reset_index(name='SCR')

scr_agent = get_precise_scr('agent')
scr_type = get_precise_scr('type')

# --- 4. FUNZIONE DI CALCOLO METRICHE (Modificata per non filtrare internamente) ---
def compute_final_metrics(df_grouped, grouping_col, scr_df):
    rep = df_grouped.groupby(grouping_col).agg({
        'is_merged': 'mean',
        'time_hrs': 'median',
        'time_hrs_clipped': 'mean',
        'n_comments': 'mean',
        'filename': ['median', 'mean'],
        'additions': 'sum', 'deletions': 'sum',
        'ASI_pr': 'mean', 'PCD_pr': 'mean', 'id': 'count'
    }).reset_index()

    cols = [grouping_col] if isinstance(grouping_col, str) else grouping_col
    rep.columns = cols + ['acc_rate', 't_med', 't_mean', 'c_avg', 'f_med', 'f_avg', 'add_sum', 'del_sum', 'ASI', 'PCD', 'sample']

    rep['i_struct'] = (rep['f_avg'] / (rep['f_med'] + 0.001))
    rep['i_proc'] = (rep['t_mean'] / (rep['t_med'] + 0.001))
    rep['GRS'] = 1 / np.log1p(rep['i_struct'] + rep['i_proc'] + 1)

    for c in ['i_struct', 'i_proc']:
        limit = rep[c].quantile(0.95)
        temp_col = rep[c].clip(upper=limit)
        min_v, max_v = temp_col.min(), temp_col.max()
        rep[f'{c}_n'] = (temp_col - min_v) / (max_v - min_v + 0.001)

    rep['CDI'] = (rep['i_struct_n'] + rep['i_proc_n']) / 2
    rep['SFI'] = (rep['c_avg'] / (rep['f_avg'] + 0.001))
    rep['ASR'] = (rep['acc_rate'] / (rep['t_med'] + 0.001))
    rep['ACE'] = (rep['del_sum'] / (rep['add_sum'] + 1))

    join_col = 'agent' if 'agent' in scr_df.columns and 'agent' in rep.columns else 'type'
    rep = rep.merge(scr_df, on=join_col, how='left').fillna(0)

    final_cols = cols + ['CDI', 'GRS', 'ASR', 'SFI', 'ACE', 'ASI', 'PCD', 'SCR', 'sample']
    return rep[final_cols].sort_values(by='CDI', ascending=False).round(2)

# --- 5. GENERAZIONE REPORT CON SEPARAZIONE ---
print("Generazione Report Agente-Tipo...")
# 1. Calcoliamo tutto senza filtri (soglia 1)
full_data = compute_final_metrics(df_merged, ['type', 'agent'], scr_agent)

# 2. Separiamo i dati robusti da quelli scarsi
report_principale = full_data[full_data['sample'] >= ROBUST_THRESHOLD].copy()
appendice_scarsi = full_data[(full_data['sample'] < ROBUST_THRESHOLD) & (full_data['sample'] >= MIN_ABSOLUTE)].copy()

# 3. Salvataggio file Agente-Tipo
report_principale.to_csv(os.path.join(reports_dir, "report_completo_agente_tipo.csv"), index=False)
appendice_scarsi.to_csv(os.path.join(reports_dir, "appendice_bassa_numerosita.csv"), index=False)

# 4. Generazione Report Aggregati (solitamente qui il sample è alto, quindi usiamo ROBUST_THRESHOLD)
report_agenti = compute_final_metrics(df_merged, 'agent', scr_agent)
report_agenti[report_agenti['sample'] >= ROBUST_THRESHOLD].to_csv(os.path.join(reports_dir, "report_aggregato_agenti.csv"), index=False)

report_tipi = compute_final_metrics(df_merged, 'type', scr_type)
report_tipi[report_tipi['sample'] >= ROBUST_THRESHOLD].to_csv(os.path.join(reports_dir, "report_aggregato_tipologia_task.csv"), index=False)

print(f"\n✅ OPERAZIONE COMPLETATA!")
print(f"-> Report principale ({ROBUST_THRESHOLD}+ sample) salvato.")
print(f"-> Appendice (1-{ROBUST_THRESHOLD-1} sample) salvata separatamente.")

Caricamento dataset principale...
Integrazione metadati granulari...
Calcolo SCR granulare (Media delle performance sui repository)...
Generazione Report Agente-Tipo...

✅ OPERAZIONE COMPLETATA!
-> Report principale (10+ sample) salvato.
-> Appendice (1-9 sample) salvata separatamente.
